# Inheritance & Polymorphism

## 1. Single Inheritance & super()

Inheritance allows you to create a new class (the Child or Subclass) that takes on all the attributes and methods of an existing class (the Parent or Base Class).

This is about code reuse and modeling "Is-A" relationships (e.g., A Dog is an Animal, a Manager is an Employee).

In [ ]:
# The Parent Class (Base Class)
class Employee:
    company = "Tekion Corp"  # Shared class attribute

    def __init__(self, name, id_number):
        self.name = name
        self.id_number = id_number

    def get_details(self):
        return f"{self.name} (ID: {self.id_number})"

# The Child Class (Subclass) inherits from Employee
class Developer(Employee):
    # We want to add a new attribute (language) that Employee doesn't have.
    def __init__(self, name, id_number, language):
        # 1. We must call the Parent's __init__ to set up name and id_number.
        # We use super() to refer to the Parent class.
        super().__init__(name, id_number)
        
        # 2. Then we set up the unique Developer attributes.
        self.language = language

    # Method Overriding: We replace the Parent's version of this method
    # with a version specific to the Child class.
    def get_details(self):
        # We can still use the parent's logic if we want...
        base_details = super().get_details()
        # ...and add our own stuff to it.
        return f"{base_details} - Writes in {self.language}"

# Let's test it out
dev = Developer("Shubham", "E105", "Python")

print(dev.company)       # Inherited from Employee (Tekion Corp)
print(dev.get_details()) # Overridden in Developer (Shubham (ID: E105) - Writes in Python)

### The Role of super():
super() is a built-in function that returns a proxy object allowing you to call methods of the parent class. It is almost always used inside the child's __init__ to ensure the parent's setup code runs before the child adds its own specific data.

## 2. Polymorphism, Duck Typing, and EAFP
Polymorphism literally means "many forms." In OOP, it means that different objects can respond to the same method call in their own specific way.

In languages like Java, polymorphism is tightly tied to inheritance. You must declare that a Dog implements the Animal interface to call .speak() on it safely.

Python doesn't care about inheritance trees for polymorphism. Python uses Duck Typing.
"If it walks like a duck and it quacks like a duck, then it must be a duck."

In [3]:
class PDFParser:
    def parse(self):
        return "Extracting text from PDF..."

class CSVParser:
    def parse(self):
        return "Reading rows from CSV..."

# This class has NO inheritance relation to the other two
class JSONParser: 
    def parse(self):
        return "Parsing JSON keys and values..."

# Polymorphism in action
def process_document(parser):
    # Python doesn't check if 'parser' inherits from a BaseParser class.
    # It only cares: "Does this object have a .parse() method?"
    print(parser.parse())

parsers = [PDFParser(), CSVParser(), JSONParser()]

for p in parsers:
    process_document(p)

Extracting text from PDF...
Reading rows from CSV...
Parsing JSON keys and values...


```text
If you pass an object to process_document() that doesn't have a .parse() method, Python will throw an AttributeError at runtime.

This leads to a core Python design philosophy: **EAFP (Easier to Ask for Forgiveness than Permission)**. Instead of checking if an object is the "right type" before calling a method (Look Before You Leap), Python developers prefer to just call the method and catch the exception if it fails.


In [4]:
# The Pythonic EAFP way
def process_document_safely(parser):
    try:
        print(parser.parse())
    except AttributeError:
        print(f"Error: {type(parser).__name__} does not support parsing.")

## 3. Abstract Base Classes (ABCs)

Duck Typing is great for flexibility, but sometimes you are building a large system (like a plugin architecture or a complex backend service) and you need strict rules. You want to guarantee that any new Parser class created by your team absolutely has a .parse() method, otherwise, the program should crash immediately.

This is where Abstract Base Classes come in.

An ABC is a blueprint for other classes. It allows you to define methods that must be implemented by any child class. You cannot instantiate an ABC directly.

In [ ]:
from abc import ABC, abstractmethod

# 1. Inherit from ABC to make this an Abstract Base Class
class DocumentParser(ABC):
     
    # 2. Use the @abstractmethod decorator to enforce a rule
    @abstractmethod
    def parse(self, file_path):
        """Child classes MUST implement this method."""
        pass 
        
    def common_utility(self):
        # ABCs can also have normal, implemented methods that children inherit
        return "Opening file..."

# This will crash! You cannot instantiate an ABC.
# parser = DocumentParser() 
# TypeError: Can't instantiate abstract class DocumentParser with abstract method parse

# Let's create a child class
class XMLParser(DocumentParser):
    # If we forget to implement parse(), Python will throw an error 
    # the moment we try to create an XMLParser instance.
    
    def parse(self, file_path):
        return f"Parsing XML from {file_path}"

xml_parser = XMLParser() # This works perfectly!

### Why use ABCs?
They are fantastic for system design. If you are building a payment processing system in Python, you might create an AbstractPaymentGateway with @abstractmethod def process_charge(). Whether your team builds a StripeGateway or a PayPalGateway, the ABC guarantees they both implement the exact methods the rest of your system expects.